# 01. Model Preparation (Refactored)
이 노트북은 **AUROC / AUPRC를 최대한 높일 수 있는 학습 세팅**을 위해,
- **누수 방지 (Group split: subject_id)**
- **Group-level stratification (양성 환자 비율 유지)**
- **불균형 가중치(scale_pos_weight) 산출**
- **학습용/평가용 데이터 저장 규격 통일**
을 포함해 전처리된 feature 테이블을 train/val/test로 분할합니다.

> ⚠️ Sliding window 샘플이 많기 때문에, **환자 단위(subject_id)로 분할**하지 않으면 AUROC/AUPRC가 과대평가될 수 있습니다.


In [ ]:
import os, json, hashlib
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

# ==============================================================================
# 설정
# ==============================================================================
INPUT_DIR = "../../data-pipeline/data/processed"
INPUT_FILE = "features_final.csv"

# ✅ 재현성 (팀원 모두 동일해야 함)
RANDOM_STATE = 42

# ✅ 분할 비율
TEST_SIZE = 0.15
VAL_SIZE  = 0.15  # (train+val) 중 val 비율이 아니라, 전체 중 val 비율

# ✅ 피처 선택 옵션
# 'all'     : 전체 피처 그룹(ALL_FEATURES) 사용
# 'top20'   : SHAP 상위 20개 (INPUT_DIR/top20_features.json)
# 'top10'   : SHAP 상위 10개 (INPUT_DIR/top10_features.json)
# 'custom'  : 직접 지정
FEATURE_CONFIG = "all"

print("=== 01. Model Preparation (Refactored) ===")
print(f"RANDOM_STATE={RANDOM_STATE}, TEST_SIZE={TEST_SIZE}, VAL_SIZE={VAL_SIZE}, FEATURE_CONFIG={FEATURE_CONFIG}")


## Step 1. 데이터 로드

In [ ]:
df = pd.read_csv(os.path.join(INPUT_DIR, INPUT_FILE))
print(f"✓ Loaded: {len(df):,} rows, {len(df.columns):,} cols")
for k in ["stay_id","subject_id","hadm_id"]:
    if k in df.columns:
        print(f"  - unique {k}: {df[k].nunique():,}")


## Step 2. ID / Label / Feature 정의

In [ ]:
# ==============================================================================
# ID 컬럼 (학습 제외)
# ==============================================================================
ID_COLS = [
    "stay_id",
    "subject_id",
    "hadm_id",
    "observation_hour",
    "observation_start",
    "observation_end",
]

missing_ids = [c for c in ID_COLS if c not in df.columns]
if missing_ids:
    raise ValueError(f"필수 ID 컬럼 누락: {missing_ids}")

# ==============================================================================
# 레이블 컬럼: 'next_' 포함 컬럼 전체를 기본 대상으로 사용
#   - 예: death_next_6h, vent_start_next_24h, composite_next_12h ...
# ==============================================================================
LABEL_COLS = [c for c in df.columns if "next_" in c]
if not LABEL_COLS:
    raise ValueError("LABEL_COLS를 찾지 못했습니다. (컬럼명에 'next_'가 포함되어야 합니다.)")

print(f"✓ Labels: {len(LABEL_COLS)}")

# ==============================================================================
# 피처 그룹 정의 (필요에 맞게 수정/확장)
# ==============================================================================
ALL_FEATURES = {
    "vitals":  ["hr","rr","spo2","sbp","dbp","mbp","temp"],
    "labs":    ["creatinine","wbc","platelets","potassium","sodium","lactate"],
    "gcs":     ["gcs_eye","gcs_verbal","gcs_motor","gcs_total"],
    "urine":   ["urine_ml_6h","urine_ml_kg_hr_avg","oliguria_flag"],
    "derived": ["shock_index","anchor_age","news_score","mews_score"],
    "flags":   ["lactate_missing","abga_checked","is_readmission"],
    "delta":   ["hr_delta","sbp_delta","spo2_delta","lactate_delta","gcs_total_delta"],
}

# 전체 피처 후보 리스트
all_feature_list = [c for cols in ALL_FEATURES.values() for c in cols]
print(f"✓ Feature candidates: {len(all_feature_list)}")

def load_feature_config(config: str) -> list[str]:
    if config == "all":
        feats = all_feature_list
        out_dir = os.path.join(INPUT_DIR, "all_features_refactored")
    elif config.startswith("top"):
        json_path = os.path.join(INPUT_DIR, f"{config}_features.json")
        if not os.path.exists(json_path):
            raise FileNotFoundError(f"{json_path} 가 없습니다. (SHAP 기반 상위 피처 json 필요)")
        feats = json.load(open(json_path))
        out_dir = os.path.join(INPUT_DIR, f"{config}_features_refactored")
    elif config == "custom":
        feats = [
            "lactate","mbp","sbp","hr","spo2",
            "gcs_total","urine_ml_6h",
            "news_score","mews_score","shock_index","anchor_age"
        ]
        out_dir = os.path.join(INPUT_DIR, "custom_features_refactored")
    else:
        raise ValueError(f"Unknown FEATURE_CONFIG: {config}")

    # 존재하는 피처만 사용
    feats = [c for c in feats if c in df.columns]
    os.makedirs(out_dir, exist_ok=True)
    return feats, out_dir

FEATURE_COLS, OUTPUT_DIR = load_feature_config(FEATURE_CONFIG)

print(f"✓ Selected features: {len(FEATURE_COLS)}")
print(f"✓ Output dir: {OUTPUT_DIR}")

# sanity
missing = [c for c in FEATURE_COLS if c not in df.columns]
if missing:
    raise ValueError(f"선택된 피처 중 df에 없는 컬럼이 있습니다: {missing}")


## Step 3. Group(환자) 단위 Stratified Split
AUPRC를 올리려면 **양성 환자 비율이 split마다 크게 흔들리지 않도록** 하는 게 중요합니다.

여기서는:
- group = `subject_id`
- stratify label = **'가장 대표적인 타깃(Composite 24h 우선)'** 또는 **'어떤 레이블이든 하나라도 양성'**
으로 환자 단위 stratified split을 합니다.

In [ ]:
def pick_stratify_label(df: pd.DataFrame, label_cols: list[str]) -> str:
    # 우선순위: composite_next_24h → composite_next_12h → composite_next_6h → (any-positive)
    preferred = ["composite_next_24h","composite_next_12h","composite_next_6h"]
    for p in preferred:
        if p in label_cols:
            return p
    # 없으면 가장 prevalence가 높은 label 하나를 선택 (분산 안정)
    prev = {c: df[c].mean() for c in label_cols}
    return max(prev, key=prev.get)

STRATIFY_COL = pick_stratify_label(df, LABEL_COLS)
print(f"✓ Stratify on: {STRATIFY_COL} (mean={df[STRATIFY_COL].mean():.4f})")

# 환자(그룹) 레벨로 y 만들기: 환자 내 윈도우 중 하나라도 양성이면 1
g = df.groupby("subject_id")[STRATIFY_COL].max().reset_index()
groups = g["subject_id"].to_numpy()
y_group = g[STRATIFY_COL].to_numpy().astype(int)

print(f"  - groups: {len(groups):,}")
print(f"  - positive groups: {y_group.sum():,} ({y_group.mean()*100:.2f}%)")

def stratified_group_split(groups: np.ndarray, y_group: np.ndarray, test_size: float, random_state: int):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    (tr_idx, te_idx), = sss.split(groups, y_group)
    return groups[tr_idx], groups[te_idx]

# 1) train+val vs test
trainval_groups, test_groups = stratified_group_split(groups, y_group, TEST_SIZE, RANDOM_STATE)

# 2) train vs val (val_size는 전체 기준이므로 trainval 대비 비율로 변환)
val_ratio_in_trainval = VAL_SIZE / (1.0 - TEST_SIZE)
g2 = g[g["subject_id"].isin(trainval_groups)]
train_groups, val_groups = stratified_group_split(
    g2["subject_id"].to_numpy(),
    g2[STRATIFY_COL].to_numpy().astype(int),
    val_ratio_in_trainval,
    RANDOM_STATE + 1,  # split 2는 seed를 살짝 다르게
)

print(f"✓ Split groups:")
print(f"  - Train groups: {len(train_groups):,}")
print(f"  - Val groups  : {len(val_groups):,}")
print(f"  - Test groups : {len(test_groups):,}")

# row-level split
df_train = df[df["subject_id"].isin(train_groups)].copy()
df_val   = df[df["subject_id"].isin(val_groups)].copy()
df_test  = df[df["subject_id"].isin(test_groups)].copy()

def summarize_split(name, d):
    print(f"[{name}] rows={len(d):,}, subjects={d['subject_id'].nunique():,}, stays={d['stay_id'].nunique():,}")
    print(f"  {STRATIFY_COL} pos rate: {d[STRATIFY_COL].mean()*100:.2f}%")

summarize_split("TRAIN", df_train)
summarize_split("VAL", df_val)
summarize_split("TEST", df_test)


## Step 4. 클래스 불균형 가중치(scale_pos_weight)
XGBoost/LightGBM에서 **AUPRC가 안정적으로 올라가게** 하려면, 기본적으로 양성에 더 큰 가중치를 줘야 합니다.

- `scale_pos_weight = (#neg / #pos)` 를 train 기준으로 계산
- label마다 다르게 저장

In [ ]:
scale_pos_weight = {}
for col in LABEL_COLS:
    pos = float(df_train[col].sum())
    neg = float(len(df_train) - pos)
    scale_pos_weight[col] = (neg / pos) if pos > 0 else 1.0

# 보기 좋게 상위 몇 개 출력
print("=== scale_pos_weight (train 기준) ===")
for k in sorted(scale_pos_weight.keys())[:15]:
    print(f"{k:30s} {scale_pos_weight[k]:.2f}")
print(f"... total {len(scale_pos_weight)} labels")

# 참고: prevalence
print("\n=== label prevalence (train) ===")
prev_train = {c: df_train[c].mean() for c in LABEL_COLS}
for k in sorted(prev_train, key=prev_train.get, reverse=True)[:10]:
    print(f"{k:30s} {prev_train[k]*100:.3f}%")


## Step 5. 저장 (train/val/test + feature list + split info)
학습 노트북(02_train_*)에서 바로 읽을 수 있게 저장합니다.

In [ ]:
# 저장 경로
train_path = os.path.join(OUTPUT_DIR, "train.csv")
val_path   = os.path.join(OUTPUT_DIR, "val.csv")
test_path  = os.path.join(OUTPUT_DIR, "test.csv")

# ✅ 학습에 필요한 컬럼만 저장 (ID + FEATURES + LABELS)
save_cols = ID_COLS + FEATURE_COLS + LABEL_COLS
df_train[save_cols].to_csv(train_path, index=False)
df_val[save_cols].to_csv(val_path, index=False)
df_test[save_cols].to_csv(test_path, index=False)

# 피처 목록 저장
with open(os.path.join(OUTPUT_DIR, "features.json"), "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

# split 정보 저장
split_info = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "val_size": VAL_SIZE,
    "feature_config": FEATURE_CONFIG,
    "stratify_col": STRATIFY_COL,
    "n_train_subjects": int(len(train_groups)),
    "n_val_subjects": int(len(val_groups)),
    "n_test_subjects": int(len(test_groups)),
    "hash_train_subjects": hashlib.md5((",".join(map(str, train_groups[:100]))).encode()).hexdigest(),
}
with open(os.path.join(OUTPUT_DIR, "split_info.json"), "w") as f:
    json.dump(split_info, f, indent=2)

# 가중치 저장
with open(os.path.join(OUTPUT_DIR, "scale_pos_weight.json"), "w") as f:
    json.dump(scale_pos_weight, f, indent=2)

print("✓ Saved:")
for p in [train_path, val_path, test_path]:
    print(f"  - {p}  ({os.path.getsize(p)/1024/1024:.1f} MB)")
print(f"  - {os.path.join(OUTPUT_DIR,'features.json')}")
print(f"  - {os.path.join(OUTPUT_DIR,'split_info.json')}")
print(f"  - {os.path.join(OUTPUT_DIR,'scale_pos_weight.json')}")


## Next: AUROC/AUPRC 올리는 학습 팁 (02_train 노트북에서 반영)
- `eval_metric=['auc','aucpr']` + `early_stopping_rounds`
- `scale_pos_weight` 적용
- **PR-AUC 최적화**를 위해 early stopping 기준을 `aucpr`에 두기
- 가능하면 `max_depth` 낮추고 `min_child_weight`/`reg_lambda`/`subsample` 조정(과적합 방지)
- `GroupKFold` 기반 CV로 튜닝하면 더 안정적으로 개선
